# Beschreibung: 

# Importe:

In [1]:
import pandas as pd

# Funktionen:

In [2]:
def discretize(df, cols, bins=4):
    """
    Numerische Werte werden in diskrete Kategorien überführt.
    Input:
        • df: Zu diskreditierende Pandas-DataFrame
        • cols: Liste numerischer Spalten, die kategorisiert werden sollen
        • bins: Anzahl der Diskretisierungsintervalle (Quantils-binning)
    Output:
        • df: Ein neues DataFrame mit zusätzlichen Spalten:
            Original:          New column:
            "Total_Score"  →   "Total_Score_disc"
            "Age"          →   "Age_disc"
        
    Beispiel:
        Werte 0–100 werden bei bins=4() zu:
        0 → niedrigstes Quartil
        1 → zweites Quartil
        2 → drittes Quartil
        3 → höchstes Quartil
    """
    df = df.copy()
    for col in cols:
        if df[col].nunique() <= 1:
            df[col+"_disc"] = 0
        else:
            df[col+"_disc"] = pd.qcut(
                df[col].rank(method="first"), 
                q=bins, 
                labels=False, 
                duplicates="drop"
            )
    return df


def indiscernibility(df, attrs):
    """
    Bildet Äquivalenzklassen der Indiscernibility Relation 'IND(P)' wieder.
    Input:
        • df: ein DataFrame (diskretisiert)
        • attrs: Liste von Attributen, nach denen Objekte verglichen werden
    Output:
        • list: Eine Liste von Listen, wo jede innere Liste eine Äquivalenzklasse [x]P ist.

    Beispiel:
        Input:
            • attrs = ["Age_disc", "StudyTime_disc"]
            df: ...
            Objekt 1: (2, 0)
            Objekt 2: (2, 0)
            Objekt 3: (1, 3)
                ...
        Output:

        [
          [0, 4, 10],     # diese drei Objekte sind ununterscheidbar
          [1, 2],         # diese beiden ebenfalls - ist auch im beispielhaften Input zu sehen.
          [3],            # einzelnes Objekt
          ...
        ]
    """
    groups = {}
    for i, row in df[attrs].iterrows():
        key = tuple(row.tolist())
        groups.setdefault(key, []).append(i)
    return list(groups.values())


def dependency(df, attrs, decision):
    """
    • Berechnet POS_C(D)
    • Zählt, wie viele Objekte eindeutig klassifiziert werden können
    • Dividiert durch Anzahl aller Objekte
    
    Input:
        • df: Information System (diskret)
        • attrs: Konditionsattribute C
        • decision: Entscheidungsattribut D
    Output:
        • Ein float zwischen 0 und 1

    Für die Attribute C:
        • indiscernibility(C) liefert Äquivalenzklassen
            -> Jede Klasse wird geprüft:
                wenn alle Objekte dieselbe Entscheidung haben → Klasse gehört zur positiven Region


    Ein float zwischen 0 und 1:
        • γ = 1 → perfekte Klassifikation
        • γ = 0 → keine Klassifikation möglich
        • 0 < γ < 1 → teils eindeutig, teils unsicher
    """
    if not attrs:
        return 0
    pos = 0
    U = len(df)
    for block in indiscernibility(df, attrs):
        if df.loc[block, decision].nunique() == 1:
            pos += len(block)
    return pos / U


def quick_reduct(df, attrs, decision):
    """
    Was wird gemacht:
        • greedy Auswahl von Attributen
        • Maximiert schrittweise den Dependency Degree γ
        • Endet:
            • Kein Attribut γ verbessern kann. Oder
            • γ voll ist.
    Ursprung: Pawlak (1982) – Grunddefinition Reduct und Shenoi & Yao (1994), Jensen (1998) – QuickReduct-Algorithmus
    
    Input:
        • df: Diskretes IS
        • attrs: Alle konditionalen Attribute
        • decision: Die Zielvariable
    Output:
        • list: Eine Liste von Attributen z.B ['Age_disc'], welches das (hoffentlich:)) minimale notwendige Attributset ist.
    """
    R = []
    gamma_star = dependency(df, attrs, decision)
    gamma_R = 0

    while gamma_R < gamma_star - 1e-12:
        best_attr = None
        best_gamma = gamma_R

        for a in attrs:
            if a in R: continue
            g = dependency(df, R+[a], decision)
            if g > best_gamma + 1e-12:
                best_gamma = g
                best_attr = a

        if best_attr is None:
            break

        R.append(best_attr)
        gamma_R = best_gamma

    return R


def induce_rules(df, reduct, decision):
    """
    Aus jeder Äquivalenzklasse [x]_R:
        Wenn alle Objekte dieselbe Entscheidung haben → Regel: IF (Attribut1 = Wert1 ∧ …) THEN (Decision = Klasse)
    Regeln entstehen genau aus den reinen Blöcken der Reduct-Indiscernibility. (Das entspricht exakt Pawlaks Methode.)
    
    Input:
        • df: Diskretes IS
        • reduct: Vom QuickReduct ausgewählte Atributenset
        • decision: Entscheidungsattribut
    Output:
        • Eine Liste von Regeln

    Beispielhafte Ausgabe:
        {
          'premise': {'Total_Score_disc': 3},
          'decision': 'A',
          'support': 1250
        }
        Dabei gilt es folgendermaßen zu lesen: "IF (premise) THEN (Decision)" und der support gibt die Anzahl der Objekte in dieser Klasse
    """
    rules = []
    for block in indiscernibility(df, reduct):
        decs = df.loc[block, decision].unique()
        if len(decs)==1:
            rules.append({
                "premise": {a: df.loc[block[0], a] for a in reduct},
                "decision": decs[0],
                "support": len(block),
            })
    return rules


# Daten laden:

In [3]:
biased = pd.read_csv("./Kaggle_Daten/Student_Performance_Behavior_Dataset/Students_Grading_Dataset_Biased.csv")
unbiased = pd.read_csv("./Kaggle_Daten/Student_Performance_Behavior_Dataset/Students_Performance_Dataset.csv")

print(biased.shape)
print(unbiased.shape)

(5000, 23)
(5000, 23)


# Ausführung

### Vorbereitung: (Datenaufbereitung)

In [4]:
# Als Entscheidung wird folgendes genutzt:
decision_attr = "Grade"

In [5]:
# Numerische Spalten finden:
num_cols = biased.select_dtypes(include=["int64","float64"]).columns

In [21]:
# Diskretisierung der Daten:
biased_disc = discretize(biased, num_cols, bins=4)
unbiased_disc = discretize(unbiased, num_cols, bins=4)

In [16]:
# Konditionsattribute
cond_attrs_o_num = [
    col for col in biased_disc.columns 
    if col.endswith("_disc") and col != "Grade"
]
print(cond_attrs_o_num)

['Age_disc', 'Attendance (%)_disc', 'Midterm_Score_disc', 'Final_Score_disc', 'Assignments_Avg_disc', 'Quizzes_Avg_disc', 'Participation_Score_disc', 'Projects_Score_disc', 'Total_Score_disc', 'Study_Hours_per_Week_disc', 'Stress_Level (1-10)_disc', 'Sleep_Hours_per_Night_disc']


In [17]:
# Konditionsattribute
cond_attrs_all = []
for col in biased_disc.columns:
    if col != decision_attr:
        # diskretisierte numerische Werte (z. B. Total_Score_disc)
        if col.endswith("_disc"):
            cond_attrs_all.append(col)
        # direkt kategorische Werte (Gender, Department etc.)
        elif biased_disc[col].dtype == "object":
            cond_attrs_all.append(col)
            
cond_attrs_all.remove('Student_ID') # Das ist ein Identifier, daher muss es raus.
cond_attrs_all.remove('Email') # Es gilt hier dasselbe

print(cond_attrs_all)

['First_Name', 'Last_Name', 'Gender', 'Department', 'Extracurricular_Activities', 'Internet_Access_at_Home', 'Parent_Education_Level', 'Family_Income_Level', 'Age_disc', 'Attendance (%)_disc', 'Midterm_Score_disc', 'Final_Score_disc', 'Assignments_Avg_disc', 'Quizzes_Avg_disc', 'Participation_Score_disc', 'Projects_Score_disc', 'Total_Score_disc', 'Study_Hours_per_Week_disc', 'Stress_Level (1-10)_disc', 'Sleep_Hours_per_Night_disc']


### Datenbetrachtung:

#### cond_attrs_o_num: Datensatz mit nur diskretisierten numerischen Werte

In [18]:
# Redukte
biased_reduct_o_num = quick_reduct(biased_disc, cond_attrs_o_num, "Grade")
unbiased_reduct_o_num = quick_reduct(unbiased_disc, cond_attrs_o_num, "Grade")

print("Biased Reduct nur numerische Werte:\n", biased_reduct_o_num)
print("\nUnbiased Reduct nur numerische Werte:\n", unbiased_reduct_o_num)

Biased Reduct nur numerische Werte:
 ['Assignments_Avg_disc', 'Attendance (%)_disc']

Unbiased Reduct nur numerische Werte:
 ['Total_Score_disc']


In [19]:
rules_unbiased_o_num = induce_rules(unbiased_disc, unbiased_reduct_o_num, "Grade")

for r in rules_unbiased_o_num[:10]:
    print(r)

{'premise': {'Total_Score_disc': np.int64(2)}, 'decision': 'C', 'support': 1250}


In [20]:
biased_pass_reduct_o_num = quick_reduct(biased_disc, cond_attrs_o_num, "Pass")
unbiased_pass_reduct_o_num = quick_reduct(unbiased_disc, cond_attrs_o_num, "Pass")

biased["Pass"] = (biased["Grade"].isin(["A","B"])).astype(int)
unbiased["Pass"] = (unbiased["Grade"].isin(["A","B"])).astype(int)

rules_pass_biased_o_num = induce_rules(biased_disc, biased_pass_reduct_o_num, "Pass")
rules_pass_unbiased_o_num = induce_rules(unbiased_disc, unbiased_pass_reduct_o_num, "Pass")

#### cond_attrs_all: Datensatz mit nicht nur diskretisierten numerischen Werten sondern auch direkt kategorische Werte (Gender, Department etc.)

In [22]:
# Redukte
biased_reduct_all = quick_reduct(biased_disc, cond_attrs_all, "Grade")
unbiased_reduct_all = quick_reduct(unbiased_disc, cond_attrs_all, "Grade")

print("Biased Reduct alle Werte:\n", biased_reduct_all)
print("\nUnbiased Reduct alle Werte:\n", unbiased_reduct_all)

Biased Reduct alle Werte:
 ['Assignments_Avg_disc', 'Attendance (%)_disc']

Unbiased Reduct alle Werte:
 ['Total_Score_disc']


In [23]:
rules_unbiased_all = induce_rules(unbiased_disc, unbiased_reduct_all, "Grade")

for r in rules_unbiased_all[:10]:
    print(r)

{'premise': {'Total_Score_disc': np.int64(2)}, 'decision': 'C', 'support': 1250}


In [24]:
biased_pass_reduct_all = quick_reduct(biased_disc, cond_attrs_all, "Pass")
unbiased_pass_reduct_all = quick_reduct(unbiased_disc, cond_attrs_all, "Pass")

biased["Pass"] = (biased["Grade"].isin(["A","B"])).astype(int)
unbiased["Pass"] = (unbiased["Grade"].isin(["A","B"])).astype(int)

rules_pass_biased_all = induce_rules(biased_disc, biased_pass_reduct_all, "Pass")
rules_pass_unbiased_all = induce_rules(unbiased_disc, unbiased_pass_reduct_all, "Pass")

In [33]:
#print(biased)
#print(unbiased)
print(biased.columns)
print(unbiased.columns)

Index(['Student_ID', 'First_Name', 'Last_Name', 'Email', 'Gender', 'Age',
       'Department', 'Attendance (%)', 'Midterm_Score', 'Final_Score',
       'Assignments_Avg', 'Quizzes_Avg', 'Participation_Score',
       'Projects_Score', 'Total_Score', 'Grade', 'Study_Hours_per_Week',
       'Extracurricular_Activities', 'Internet_Access_at_Home',
       'Parent_Education_Level', 'Family_Income_Level', 'Stress_Level (1-10)',
       'Sleep_Hours_per_Night', 'Pass'],
      dtype='object')
Index(['Student_ID', 'First_Name', 'Last_Name', 'Email', 'Gender', 'Age',
       'Department', 'Attendance (%)', 'Midterm_Score', 'Final_Score',
       'Assignments_Avg', 'Quizzes_Avg', 'Participation_Score',
       'Projects_Score', 'Total_Score', 'Grade', 'Study_Hours_per_Week',
       'Extracurricular_Activities', 'Internet_Access_at_Home',
       'Parent_Education_Level', 'Family_Income_Level', 'Stress_Level (1-10)',
       'Sleep_Hours_per_Night', 'Pass'],
      dtype='object')


# Resultate

In [28]:
print("Nur numerische Werte:")
print("Biased Reduct:\n", biased_reduct_o_num)
print("\nUnbiased Reduct:\n", unbiased_reduct_o_num)

print("\nPass Reduct (biased):", biased_pass_reduct_o_num)
print("Pass Reduct (unbiased):", unbiased_pass_reduct_o_num)

#print("Pass Rules (biased):", rules_pass_biased_o_num)
#print("Pass Rules (unbiased):", rules_pass_unbiased_o_num)

print("\n")
print("Alle Werte:")
print("Biased Reduct:\n", biased_reduct_all)
print("\nUnbiased Reduct:\n", unbiased_reduct_all)

print("\nPass Reduct (biased):", biased_pass_reduct_all)
print("Pass Reduct (unbiased):", unbiased_pass_reduct_all)

#print("Pass Rules (biased):", rules_pass_biased_all)
#print("Pass Rules (unbiased):", rules_pass_unbiased_all)

Nur numerische Werte:
Biased Reduct:
 ['Assignments_Avg_disc', 'Attendance (%)_disc']

Unbiased Reduct:
 ['Total_Score_disc']

Pass Reduct (biased): ['Attendance (%)_disc', 'Assignments_Avg_disc']
Pass Reduct (unbiased): ['Total_Score_disc']


Alle Werte:
Biased Reduct:
 ['Assignments_Avg_disc', 'Attendance (%)_disc']

Unbiased Reduct:
 ['Total_Score_disc']

Pass Reduct (biased): ['Attendance (%)_disc', 'Assignments_Avg_disc', 'Department', 'First_Name', 'Last_Name', 'Stress_Level (1-10)_disc', 'Study_Hours_per_Week_disc', 'Total_Score_disc', 'Parent_Education_Level']
Pass Reduct (unbiased): ['Total_Score_disc']
